<a href="https://colab.research.google.com/github/JuliusdeGroot/gpu/blob/main/gpu_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPU Programming

To use CUDA, you need to enable a GPU runtime. Here's how you can change your Colab runtime to use a GPU:

- Go to the "Runtime" menu at the top of the Colab interface.
- Select "Change runtime type".
- In the "Hardware accelerator" dropdown menu, choose a "GPU".
- Click "Save".

Now run the following blocks, by clicking the 'Play' icon or setting focus and pressing Shift-Enter, to set things up and run the add_vec_gpu3 example.

In [15]:
# Install NVIDIA CUDA Toolkit, this can take a while.
!apt-get -qq install nvidia-cuda-toolkit
print('NVIDIA CUDA Toolkit installation complete. Please run the next cell to verify.')

NVIDIA CUDA Toolkit installation complete. Please run the next cell to verify.


In [16]:
# Check the nvcc version to confirm CUDA installation
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


### Run the following blocks to add the files

- utils.h         : with some vector helper functions
- gpu_utils.h     : with some CUDA helper functions
- add_vec_gpu3.cu : the vector addition example  


In [17]:
%%writefile utils.h
#ifndef __UTILS_H__
#define __UTILS_H__

#include <vector>
#include <random>
#include <iostream>
#include <cfloat>
using namespace std;

vector<float> get_random_vector(int n) {
    vector<float> v(n);
    for (int i = 0; i < n; i++)
        v[i] = static_cast<float>(rand()) / RAND_MAX;
    return v;
}

ostream& operator<<(ostream& os, const vector<float>& v) {
    for (size_t i = 0; i < v.size(); i++)
        os << v[i] << ' ';
    return os;
}

#endif

Overwriting utils.h


In [18]:
%%writefile gpu_utils.h
#ifndef __GPU_UTILS_H__
#define __GPU_UTILS_H__

#include <cuda_runtime.h>
#include <iostream>
#include <stdio.h>
#include "utils.h"

static void HandleError( cudaError_t err,
                         const char *file,
                         int line ) {
    if (err != cudaSuccess) {
        printf( "%s in %s at line %d\n", cudaGetErrorString( err ),
                file, line );
        exit( EXIT_FAILURE );
    }
}
#define HANDLE_ERROR( err ) (HandleError( err, __FILE__, __LINE__ ))

int get_cuda_device_count() {
    int count;
    HANDLE_ERROR(cudaGetDeviceCount(&count));
    return count;
}

cudaDeviceProp get_cuda_device_properties(int device_id) {
    cudaDeviceProp prop;
    HANDLE_ERROR(cudaGetDeviceProperties(&prop, device_id));
    return prop;
}

void print_cuda_device_properties(const cudaDeviceProp& prop) {
    std::cout<< "Device  prop.name: " << prop.name << std::endl;
    std::cout<< "  Compute capability  prop.major: " << prop.major << ", prop.minor: " << prop.minor << std::endl;
    std::cout<< "  Clock rate  prop.clockRate: " << prop.clockRate << std::endl;
    std::cout<< "  Multiprocessor count  prop.multiProcessorCount: " << prop.multiProcessorCount << std::endl;
    std::cout<< "  Total global memory  prop.totalGlobalMem: " << prop.totalGlobalMem << " bytes" << std::endl;
    std::cout<< "  Total constant memory  prop.totalConstMem: " << prop.totalConstMem << " bytes" << std::endl;
    std::cout<< "  Shared memory per block  prop.sharedMemPerBlock: " << prop.sharedMemPerBlock << " bytes" << std::endl;
    std::cout<< "  Shared memory per multiprocessor  prop.sharedMemPerMultiprocessor: " << prop.sharedMemPerMultiprocessor << " bytes" << std::endl;
    std::cout<< "  Registers per block  prop.regsPerBlock: " << prop.regsPerBlock << std::endl;
    std::cout<< "  Registers per multiprocessor  prop.regsPerMultiprocessor: " << prop.regsPerMultiprocessor << std::endl;
    std::cout<< "  Warp size  prop.warpSize: " << prop.warpSize << std::endl;
    std::cout<< "  Max threads per block  prop.maxThreadsPerBlock: " << prop.maxThreadsPerBlock << std::endl;
    std::cout<< "  Max threads per multiprocessor  prop.maxThreadsPerMultiProcessor: " << prop.maxThreadsPerMultiProcessor << std::endl;
    std::cout<< "  Max thread dimensions  prop.maxThreadsDim: (" << prop.maxThreadsDim[0] << ", " << prop.maxThreadsDim[1] << ", " << prop.maxThreadsDim[2] << ")" << std::endl;
    std::cout<< "  Max grid dimensions  prop.maxGridSize: (" << prop.maxGridSize[0] << ", " << prop.maxGridSize[1] << ", " << prop.maxGridSize[2] << ")" << std::endl;
    std::cout<< "  Concurrent kernels  prop.concurrentKernels: " << prop.concurrentKernels << std::endl;
    std::cout<< "  Async engine count  prop.asyncEngineCount: " << prop.asyncEngineCount << std::endl;
    std::cout<< "  Kernel execution timeout enabled  prop.kernelExecTimeoutEnabled: " << prop.kernelExecTimeoutEnabled << std::endl;
}


#endif

Overwriting gpu_utils.h


In [19]:
%%writefile add_vec_gpu3.cu
#include "gpu_utils.h"

__global__ void add_vec_gpu(const float* d_v1, const float* d_v2, float* d_add, int n) {
    for (int i = threadIdx.x + blockIdx.x * blockDim.x;
        i < n;
        i += blockDim.x * gridDim.x)
        d_add[i] = d_v1[i] + d_v2[i];
}

int main() {
    //srand(time(nullptr));
    const int n = 18;
    vector<float> h_v1 = get_random_vector(n);
    vector<float> h_v2 = get_random_vector(n);
    vector<float> h_add(n);

    cout << "v1: " << h_v1 << endl;
    cout << "v2: " << h_v2 << endl;

    float *d_v1, *d_v2, *d_add;
    // Allocate device memory
    HANDLE_ERROR(cudaMalloc((void**)&d_v1, sizeof(float) * n));
    HANDLE_ERROR(cudaMalloc((void**)&d_v2, sizeof(float) * n));
    HANDLE_ERROR(cudaMalloc((void**)&d_add, sizeof(float) * n));

    // Transfer data from host to device memory
    HANDLE_ERROR(cudaMemcpy(d_v1, h_v1.data(), sizeof(float) * n, cudaMemcpyHostToDevice));
    HANDLE_ERROR(cudaMemcpy(d_v2, h_v2.data(), sizeof(float) * n, cudaMemcpyHostToDevice));

    add_vec_gpu<<<2, 4>>>(d_v1, d_v2, d_add, n);
    HANDLE_ERROR(cudaGetLastError());
    HANDLE_ERROR(cudaDeviceSynchronize());

    // Transfer data back to host memory
    HANDLE_ERROR(cudaMemcpy(h_add.data(), d_add, sizeof(float) * n, cudaMemcpyDeviceToHost));

    // Deallocate device memory
    HANDLE_ERROR(cudaFree(d_v1));
    HANDLE_ERROR(cudaFree(d_v2));
    HANDLE_ERROR(cudaFree(d_add));

    cout << "addition: " << h_add << endl;
}

Overwriting add_vec_gpu3.cu


In [20]:
# Now compile the CUDA program using nvcc
# -o specifies the output executable name
!nvcc add_vec_gpu3.cu -o add_vec_gpu3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [21]:
# Run the compiled CUDA program
!./add_vec_gpu3

v1: 0.840188 0.394383 0.783099 0.79844 0.911647 0.197551 0.335223 0.76823 0.277775 0.55397 0.477397 0.628871 0.364784 0.513401 0.95223 0.916195 0.635712 0.717297 
v2: 0.141603 0.606969 0.0163006 0.242887 0.137232 0.804177 0.156679 0.400944 0.12979 0.108809 0.998924 0.218257 0.512932 0.839112 0.61264 0.296032 0.637552 0.524287 
addition: 0.98179 1.00135 0.7994 1.04133 1.04888 1.00173 0.491902 1.16917 0.407565 0.662779 1.47632 0.847128 0.877717 1.35251 1.56487 1.21223 1.27326 1.24158 
